# MSTAR SAR Deep Learning Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohitgit1/Target-Detection-in-MSTAR-Images/blob/main/notebooks/02_deep_learning_training.ipynb)

Train state-of-the-art **A-ConvNet** or **ResNet-18** on the MSTAR benchmark with PyTorch, cosine annealing learning rate schedule, data augmentation, and confusion matrix evaluation.

In [ ]:
import sys
!pip install -q torch torchvision scikit-learn matplotlib
sys.path.append('..')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix

from mstar_atr.constants import CLASSES
from mstar_atr.data.downloader import prepare_mstar_dataset
from mstar_atr.training.trainer import train_mstar_model, load_checkpoint, evaluate_model
from mstar_atr.data.dataset import get_mstar_dataloaders

## 1. Prepare Dataset
Ensure MSTAR dataset splits (train and test) are ready.

In [ ]:
data_dir = prepare_mstar_dataset(target_dir='../data/mstar', verbose=True)
print(f'Using dataset at: {data_dir}')

## 2. Train A-ConvNet Model
Train using AdamW and Cosine Annealing scheduler.

In [ ]:
results = train_mstar_model(
    data_dir=data_dir,
    model_name='aconvnet',
    epochs=15,
    batch_size=16,
    lr=1e-3,
    output_dir='../checkpoints',
    verbose=True
)

## 3. Plot Learning Curves

In [ ]:
h = results['history']
epochs_range = range(1, len(h['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_range, h['train_loss'], label='Train Loss', marker='o')
ax1.plot(epochs_range, h['val_loss'], label='Val Loss', marker='s')
ax1.set_title('Cross Entropy Loss')
ax1.set_xlabel('Epoch')
ax1.grid(True)
ax1.legend()

ax2.plot(epochs_range, [acc * 100 for acc in h['train_acc']], label='Train Acc', marker='o')
ax2.plot(epochs_range, [acc * 100 for acc in h['val_acc']], label='Val Acc', marker='s')
ax2.set_title('Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.grid(True)
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Evaluate & Confusion Matrix

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, _ = load_checkpoint(results['best_checkpoint'], device=device)
_, test_loader = get_mstar_dataloaders(data_dir=data_dir, batch_size=16)

loss, acc, preds, targets = evaluate_model(model, test_loader, device=device)
print(f'Test Accuracy: {acc * 100:.2f}%')

cm = confusion_matrix(targets, preds)
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('MSTAR 10-Class Confusion Matrix')
fig.colorbar(im)
tick_marks = np.arange(len(CLASSES))
ax.set_xticks(tick_marks)
ax.set_xticklabels(CLASSES, rotation=45)
ax.set_yticks(tick_marks)
ax.set_yticklabels(CLASSES)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.show()